# Task 4: Strategic Intelligence Engine

This notebook identifies Opportunities, Risks, and Trends from our 
documents stored in ChromaDB.

Process for each category:
1. Ask ChromaDB a targeted question (retrieval)
2. Get the most relevant documents back
3. Send those documents to an LLM (Ollama running Phi-4 Mini) to summarize and structure the insight
4. Output a clean opportunity/risk/trend with evidence

This is RAG: Retrieval (ChromaDB) + Generation (LLM reasoning)

In [14]:
import json
import os
import chromadb
from sentence_transformers import SentenceTransformer
import requests
from dotenv import load_dotenv

## Step 1: Connect to ChromaDB and Ollama

In [15]:
# Connect to our existing ChromaDB collection
chroma_client = chromadb.PersistentClient(path="../storage/chromadb")
collection = chroma_client.get_collection("lufthansa_intelligence")

print(f"Connected to ChromaDB")
print(f"Documents available : {collection.count()}")

load_dotenv("../.env")

model = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2",
    token=os.getenv("HF_TOKEN")
)

# Test Ollama is running and reachable
test_response = requests.post(
    "http://localhost:11434/api/generate",
    json={"model": "phi4-mini", "prompt": "Hello", "stream": False}
)
print("Ollama connected:", test_response.json()["response"][:50])

Connected to ChromaDB
Documents available : 437


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4128.99it/s]


Ollama connected: Hello! How can I assist you today?


## Step 2: Build the Retrieval + Reasoning Function

For each category (Opportunities, Risks, Trends) we:
1. Search ChromaDB with a targeted question
2. Get the top relevant documents back
3. Send those documents to the LLM with instructions to extract structured insights
4. The LLM returns insights with supporting evidence

In [16]:
# Search ChromaDB for relevant documents
def search_documents(query, n_results=5):

    query_embedding = model.encode([query]).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=n_results)

    found_documents = []

    for metadata, text in zip(results["metadatas"][0], results["documents"][0]):
        found_documents.append({
            "title"   : metadata["title"],
            "text"    : text,
            "category": metadata["category"],
            "source"  : metadata["source"],
            "url"     : metadata["url"]
        })

    return found_documents

In [17]:
# Define specific sub-topics for each insight type, based on exam examples
risk_topics = [
    "Lufthansa competitive threats from other airlines",
    "Lufthansa regulatory changes and government policy",
    "Lufthansa negative public sentiment and customer complaints",
    "Lufthansa supply chain and operational issues"
]

opportunity_topics = [
    "Lufthansa emerging technology and AI innovation",
    "Lufthansa new markets and route expansion",
    "Lufthansa partnerships and alliances",
    "Lufthansa new products and services"
]

trend_topics = [
    "Lufthansa technology trends in aviation",
    "Lufthansa customer behaviour and travel preferences",
    "Lufthansa industry developments and market shifts"
]

print(f"Risk sub-topics        : {len(risk_topics)}")
print(f"Opportunity sub-topics : {len(opportunity_topics)}")
print(f"Trend sub-topics       : {len(trend_topics)}")

Risk sub-topics        : 4
Opportunity sub-topics : 4
Trend sub-topics       : 3


## Step 3: Reasoning Function with Ollama

This function takes the retrieved documents (which may be English or German)
and asks the local LLM (Phi-4 Mini via Ollama) to extract structured insights in English.

In [18]:
# For each sub-topic, search ChromaDB and collect unique documents
def gather_documents_for_topics(topics, n_results=3):

    all_found = []
    seen_titles = set()

    for topic in topics:
        found = search_documents(topic, n_results=n_results)

        for doc in found:
            if doc["title"] not in seen_titles:
                seen_titles.add(doc["title"])
                all_found.append(doc)

    return all_found


# Gather documents for each insight type
risk_documents        = gather_documents_for_topics(risk_topics)
opportunity_documents = gather_documents_for_topics(opportunity_topics)
trend_documents        = gather_documents_for_topics(trend_topics)

print(f"Risk documents        : {len(risk_documents)}")
print(f"Opportunity documents : {len(opportunity_documents)}")
print(f"Trend documents       : {len(trend_documents)}")

Risk documents        : 12
Opportunity documents : 11
Trend documents       : 9


## Step 4: Generate Opportunities and Trends

reuse the same `search_documents` and `analyze_with_llm` functions
for Opportunities and Trends — just with different search queries.

In [19]:
def analyze_with_llm_structured(documents, insight_type):

    combined_text = ""
    for doc in documents:
        combined_text += f"- {doc['title']} (Source: {doc['source']}): {doc['text']}\n\n"

    if insight_type == "risks":
        score_field = "severity"
    else:
        score_field = "impact"

    prompt = f"""You are a strategic analyst for Lufthansa Group.

Documents (some in German, respond in English):
{combined_text}

Identify the top 4 {insight_type} for Lufthansa, covering different angles
(not just one topic repeated).

Respond ONLY with valid JSON, nothing else, in this format:
[
  {{
    "title": "short title",
    "description": "2-3 sentence description",
    "evidence": "which document title supports this",
    "{score_field}": "High, Medium, or Low",
    "confidence_score": "a number from 1 to 10"
  }}
]"""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "phi4-mini",
            "prompt": prompt,
            "stream": False
        }
    )

    text = response.json()["response"]

    # Remove markdown code fences if present
    text = text.replace("```json", "").replace("```", "")

    # Extract only the JSON array part
    start = text.find("[")
    end = text.rfind("]") + 1
    text = text[start:end]

    return json.loads(text)


print("Generating Risks...")
risks = analyze_with_llm_structured(risk_documents, "risks")

print("Generating Opportunities...")
opportunities = analyze_with_llm_structured(opportunity_documents, "opportunities")

print("Generating Trends...")
trends = analyze_with_llm_structured(trend_documents, "trends")

print(f"\nRisks: {len(risks)}, Opportunities: {len(opportunities)}, Trends: {len(trends)}")

Generating Risks...
Generating Opportunities...
Generating Trends...

Risks: 4, Opportunities: 4, Trends: 4


In [25]:
def clean_evidence(value):
    if isinstance(value, list):
        return "; ".join(str(v) for v in value)
    return str(value)

def normalize_keys(items, score_field):
    field_aliases = {
        "title": ["title", "name"],
        "description": ["description", "summary", "details"],
        score_field: [score_field, "impact_level", "severity_level", "business_impact", "risk_level"],
        "confidence_score": ["confidence_score", "confidence", "score"],
        "evidence": ["evidence", "source", "evidence_source"]
    }

    normalized = []
    for item in items:
        item = {k.lower(): v for k, v in item.items()}
        new_item = {}
        for canonical, aliases in field_aliases.items():
            for alias in aliases:
                if alias in item:
                    new_item[canonical] = item[alias]
                    break

        new_item.setdefault("title", "Untitled")
        new_item.setdefault("description", "")
        new_item.setdefault(score_field, "Medium")
        new_item.setdefault("confidence_score", "N/A")
        new_item.setdefault("evidence", "N/A")

        new_item["evidence"] = clean_evidence(new_item["evidence"])

        normalized.append(new_item)
    return normalized

risks = normalize_keys(risks, "severity")
opportunities = normalize_keys(opportunities, "impact")
trends = normalize_keys(trends, "impact")

print("Keys normalized")
for i, t in enumerate(trends):
    print(f"Trend {i}: {list(t.keys())}")

Keys normalized
Trend 0: ['title', 'description', 'impact', 'confidence_score', 'evidence']
Trend 1: ['title', 'description', 'impact', 'confidence_score', 'evidence']
Trend 2: ['title', 'description', 'impact', 'confidence_score', 'evidence']
Trend 3: ['title', 'description', 'impact', 'confidence_score', 'evidence']


In [26]:
print("=== RISKS ===")
for r in risks:
    print(f"\n{r['title']}")
    print(f"  {r['description']}")
    print(f"  Severity: {r['severity']} | Confidence: {r['confidence_score']}/10")
    print(f"  Evidence: {r['evidence']}")

print("\n\n=== OPPORTUNITIES ===")
for o in opportunities:
    print(f"\n{o['title']}")
    print(f"  {o['description']}")
    print(f"  Impact: {o['impact']} | Confidence: {o['confidence_score']}/10")
    print(f"  Evidence: {o['evidence']}")

print("\n\n=== TRENDS ===")
for t in trends:
    print(f"\n{t['title']}")
    print(f"  {t['description']}")
    print(f"  Impact: {t['impact']} | Confidence: {t['confidence_score']}/10")
    print(f"  Evidence: {t['evidence']}")

=== RISKS ===

Increased Competition in Asia and Africa Markets
  Lufthansa faces stiff competition on lucrative long-haul routes such as those connecting Europe with growth markets like China, India, Brazil, Russia, South America, Central Asian countries, the Middle East, Eastern European Countries (EEC), Latin American regions & Caribbean island nations.
  Severity: High | Confidence: 8/10
  Evidence: Streit um Drehkreuze  Emirates and Co. setzen Lufthansa zu - Aviation.Direct

Financial Performance Decline with High Revenue but Lower Profit Margin
  Despite achieving the highest revenue in company history, Lufthansa's financial performance shows a significant drop as reflected by almost 40 percent decrease in profit margins.
  Severity: High | Confidence: 9/10
  Evidence: Lufthansa mit starkem Gewinneinbruch - ZDFheute

Customer Service and Passenger Experience Issues
  Negative feedback from customers indicates a potential decline in the quality of service offered by Lufthansa, whi

## Step 5: Get Structured JSON Output

Instead of free text, we ask the LLM to return clean JSON.
This makes it easy to display in our dashboard later.

In [27]:
from datetime import datetime

intelligence_results = {
    "risks": risks,
    "opportunities": opportunities,
    "trends": trends,
    "generated_at": datetime.now().isoformat()
}

os.makedirs("../data", exist_ok=True)
with open("../data/intelligence_results.json", "w", encoding="utf-8") as f:
    json.dump(intelligence_results, f, indent=2, ensure_ascii=False)

print("Saved intelligence results to data/intelligence_results.json")

Saved intelligence results to data/intelligence_results.json
